In [1]:
!rm -rf InkubaLM-Challenge
!git clone https://github.com/melissafasol/InkubaLM-Challenge.git
%cd InkubaLM-Challenge

Cloning into 'InkubaLM-Challenge'...
remote: Enumerating objects: 332, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 332 (delta 81), reused 49 (delta 22), pack-reused 202 (from 1)
Receiving objects: 100% (332/332), 1.15 MiB | 11.53 MiB/s, done.
Resolving deltas: 100% (208/208), done.
/content/InkubaLM-Challenge


In [2]:
!git checkout refactor-src-structure
!git pull

Branch 'refactor-src-structure' set up to track remote branch 'refactor-src-structure' from 'origin'.
Switched to a new branch 'refactor-src-structure'
Already up to date.


In [3]:
!pip install datasets
!pip install transformers datasets peft trl accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 11.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is 

In [4]:
# Cell 1: Install dependencies
!pip install -q transformers accelerate peft datasets bitsandbytes trl

# Cell 2: Imports and setup
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from datasets import load_dataset, Dataset
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, Gemma3ForConditionalGeneration

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [5]:
import sys
sys.path.append("..")  # Add parent directory to the path

import os
from typing import List
from pathlib import Path
import numpy as np

# DO NOT EDIT
# create submission file
import pandas as pd
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
)

from utils import (
    model_function,
    eval
    )

from src import (
    data_utils,
    model_utils,
    inference,
    prompts,
    evaluation,
    data_augment
    )

import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset, concatenate_datasets, Dataset, Value, DatasetDict

from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM
from peft import PeftModel, PeftConfig
from sklearn.model_selection import train_test_split

In [6]:
#os.environ["TOKENIZERS_PARALLELISM"] = "false"

from huggingface_hub import login

try:
    from google.colab import userdata

    # Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
    # vars as appropriate for your system.
    # userdata.get("HF_TOKEN") indicates that the name of the token in the Colab env is HF_TOKEN
    os.environ["hf_token_2"] = userdata.get("hf_token_2")
except:
    os.environ["hf_token_2"] = "----"

login(token=os.environ["hf_token_2"])

token = os.environ["hf_token_2"]
if token == "----":
    print("⚠️ Warning: No Hugging Face token found. Some models may not load.")
else:
    login(token=token)

In [7]:
hf_token_2 = '...' # paste your token here
os.environ["HF_TOKEN"] = hf_token_2


In [ ]:
from huggingface_hub import login
login(token=hf_token_2)  # Force login using this token


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [36]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("lelapa/InkubaLM-0.4B", token=hf_token_2)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # or add a new token
model = AutoModelForCausalLM.from_pretrained("lelapa/InkubaLM-0.4B", token=hf_token_2)


VulavulaLlamaForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


In [49]:
from peft import prepare_model_for_kbit_training
model_id = "lelapa/InkubaLM-0.4B"
model = AutoModelForCausalLM.from_pretrained(model_id,load_in_4bit=True,
    device_map="auto",trust_remote_code=True, token=hf_token_2)
model = prepare_model_for_kbit_training(model)


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.


In [50]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # <- Check this matches Inkuba arch
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)


In [28]:
import json
file_path = '/content/'

all_tasks = []
for fname in [os.path.join(file_path,"gemma_sentiment_soft_labels.jsonl"), os.path.join(file_path, "gemma_xnli_soft_labels_mc.jsonl"),  os.path.join(file_path, "gemma_mt_distilled.jsonl")]:
    with open(fname) as f:
        all_tasks.extend(json.loads(line) for line in f)

with open(os.path.join(file_path, "multitask_distill.jsonl"), 'w') as f:
    for row in all_tasks:
        f.write(json.dumps(row) + "\n")


In [29]:
import os
import json

file_path = '/content/'

# Infer language from ID or langs field
def infer_lang(example):
    if example["task"] == "translation":
        return example.get("langs", "eng-swa")
    elif "swa" in example["ID"]:
        return "swahili"
    elif "hau" in example["ID"]:
        return "hausa"
    else:
        return "unknown"

# Build prompt without <lang> tokens
def format_prompt(ex):
    instruction = ex.get("instruction", "").strip()
    input_text = ex.get("input", "").strip()
    return f"{instruction}\n{input_text}".strip() if input_text else instruction

# Load and transform data
input_files = [
    os.path.join(file_path, "gemma_sentiment_soft_labels.jsonl"),
    os.path.join(file_path, "gemma_xnli_soft_labels_mc.jsonl"),
    os.path.join(file_path, "gemma_mt_distilled.jsonl"),
]

all_tasks = []
for fname in input_files:
    with open(fname) as f:
        for line in f:
            ex = json.loads(line)
            ex["lang"] = infer_lang(ex)
            ex["prompt"] = format_prompt(ex)
            all_tasks.append(ex)

# Save to final multitask JSONL
output_path = os.path.join(file_path, "multitask_distill.jsonl")
with open(output_path, "w") as f:
    for row in all_tasks:
        f.write(json.dumps(row) + "\n")

print(f"✅ Saved multitask distillation dataset to: {output_path}")


✅ Saved multitask distillation dataset to: /content/multitask_distill.jsonl


In [32]:
from datasets import Dataset

dataset = Dataset.from_list(all_tasks)


In [100]:
def tokenize_example(example):
    enc = tokenizer(
        example["prompt"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    dec = tokenizer(
        example["output"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": dec["input_ids"],
        "soft_label": example.get("soft_label", None),
        "task": example["task"]
    }



In [101]:
from datasets import Dataset

dataset = Dataset.from_list(all_tasks)
tokenized_dataset = dataset.map(tokenize_example)

# Keep task + soft_label as is
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
    output_all_columns=True
)


Map:   0%|          | 0/900 [00:00<?, ? examples/s]

In [102]:
tokenized_dataset

Dataset({
    features: ['ID', 'task', 'instruction', 'input', 'output', 'soft_label', 'lang', 'prompt', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 900
})

In [104]:
from transformers import default_data_collator

class DistillDataCollator:
    def __call__(self, features):
        batch = default_data_collator(features)
        batch["task"] = [f.get("task", None) for f in features]
        batch["soft_label"] = [f.get("soft_label", None) for f in features]
        return batch



In [118]:
class DistillationTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        task_list = inputs.pop("task")
        soft_labels = inputs.pop("soft_label")

        outputs = model(**inputs)
        logits = outputs.logits  # [B, T, vocab]
        batch_size = logits.shape[0]
        loss = 0.0

        for i in range(batch_size):
            task = task_list[i]
            label = inputs["labels"][i]
            logit = logits[i]

            if task in ["sentiment", "xnli"] and soft_labels[i] is not None:
                # Only look at the first token logits (position 0)
                token_logits = logit[0]  # shape: [vocab]

                # Get the token IDs for the correct label space
                classes = ["positive", "neutral", "negative"] if task == "sentiment" else ["true", "neither", "false"]
                class_ids = [tokenizer(label, add_special_tokens=False)["input_ids"][0] for label in classes]

                logits_slice = token_logits[class_ids]  # shape: [3]
                soft_target = torch.tensor(list(soft_labels[i].values()), device=logits.device)

                # Mix of KL and CE
                kl = F.kl_div(logits_slice.log_softmax(dim=-1), soft_target, reduction="batchmean")
                ce = F.cross_entropy(logits_slice.unsqueeze(0), soft_target.argmax().unsqueeze(0))
                loss += 0.5 * kl + 0.5 * ce

            else:
                # Fallback to standard loss (e.g. translation)
                loss += outputs.loss if outputs.loss.ndim == 0 else outputs.loss[i]

        loss = loss / batch_size
        return (loss, outputs) if return_outputs else loss

    def _prepare_inputs(self, inputs):
        inputs = super()._prepare_inputs(inputs)
        inputs["task"] = inputs["task"]
        inputs["soft_label"] = inputs["soft_label"]
        return inputs


In [119]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./inkuba-distilled",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)


In [120]:
trainer = DistillationTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DistillDataCollator(),
    tokenizer=tokenizer
)

trainer.train()


<ipython-input-120-9e328b247991>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `DistillationTrainer.__init__`. Use `processing_class` instead.
  trainer = DistillationTrainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


ValueError: Expected input batch_size (2044) to match target batch_size (252).